# Part 1: Environment Setup

This section mounts Google Drive, sets up project directories, downloads the MedMNIST dataset from Zenodo, installs dependencies, imports all required libraries, and defines the global configuration dictionary.

In [ ]:
# ── Google Drive Setup ───────────────────────────────────────────────────────
import os
from google.colab import drive

# Mount Google Drive to access project files and saved models
drive.mount('/content/drive')

# Define top-level project and data directories
DRIVE_DIR = '/content/drive/MyDrive/CMoE'
DATA_DIR  = f'{DRIVE_DIR}/data'
os.makedirs(DATA_DIR, exist_ok=True)


Mounted at /content/drive


In [ ]:
# ── MedMNIST Dataset Download ────────────────────────────────────────────────
ZENODO_REC = "10519652"

# All 18 MedMNIST datasets to download from Zenodo
FILES = [
    "pathmnist.npz", "chestmnist.npz", "dermamnist.npz", "octmnist.npz",
    "pneumoniamnist.npz", "retinamnist.npz", "breastmnist.npz", "bloodmnist.npz",
    "tissuemnist.npz", "organamnist.npz", "organcmnist.npz", "organsmnist.npz",
    "organmnist3d.npz", "nodulemnist3d.npz", "adrenalmnist3d.npz",
    "fracturemnist3d.npz", "vesselmnist3d.npz", "synapsemnist3d.npz",
]

# Skip files already on disk; otherwise download via curl with resume support
for fname in FILES:
    dest = os.path.join(DATA_DIR, fname)
    if os.path.exists(dest):
        print(f"[skip]     {fname}")
        continue
    url = f"https://zenodo.org/records/{ZENODO_REC}/files/{fname}?download=1"
    print(f"[download] {fname}")
    os.system(f'curl -L -C - -o "{dest}" "{url}"')

print(f"\nAll files ready in: {DATA_DIR}")


[download] pathmnist.npz
[download] chestmnist.npz
[download] dermamnist.npz
[download] octmnist.npz
[download] pneumoniamnist.npz
[download] retinamnist.npz
[download] breastmnist.npz
[download] bloodmnist.npz
[download] tissuemnist.npz
[download] organamnist.npz
[download] organcmnist.npz
[download] organsmnist.npz
[download] organmnist3d.npz
[download] nodulemnist3d.npz
[download] adrenalmnist3d.npz
[download] fracturemnist3d.npz
[download] vesselmnist3d.npz
[download] synapsemnist3d.npz

All files ready in: /content/drive/MyDrive/CMoE/data


In [ ]:
requirements_path = f"{DRIVE_DIR}/requirements.txt"
!pip install -r "{requirements_path}"

Looking in links: https://data.pyg.org/whl/torch-2.10.0+cpu.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 682.4/682.4 kB 21.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 88.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 828.2/828.2 kB 103.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 306.9/306.9 kB 44.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 178.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.6/935.6 kB 114.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 

In [ ]:
# Standard library
import os
import json

# Core numeric and ML
import torch
import numpy as np
import torch.nn.functional as F

# Plotting
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Dataset and graph utilities
from torch.utils.data import Dataset
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import knn_graph, global_mean_pool, GlobalAttention, GCNConv, GATConv, SAGEConv, GINConv
from torchvision import transforms

# Evaluation metrics
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

In [ ]:
config = {
    # Data
    "file_path":     f"{DATA_DIR}/octmnist.npz",

    # Graph pipeline
    "patch_size":    4,      # height/width of each image patch (must divide image dims)
    "k_neighbors":   8,      # number of nearest neighbors per node in knn_graph

    # DataLoader
    "batch_size":    32,

    # GlobalSummary
    "pool_method":   "mean", # "mean" or "attn"
    "attn_hidden":   64,     # hidden dim of attention gate (used only when pool_method="attn")

    # ClassicalRouter
    "n_experts":     8,      # number of experts
    "router_hidden": 64,     # hidden dim of the router MLP

    # Expert GNNs
    "expert_hidden": 64,     # hidden dim inside each expert GNN
    "expert_out":    128,    # output dim of each expert (= MoE output dim)
    "top_k":         2,      # how many experts are activated per sample

    # Iterative refinement
    "n_iterations":  4,      # number of GlobalSummary → ClassicalRouter → MoE passes

    # Classifier
    "n_classes":     4,      # number of output classes
    "clf_hidden":    256,    # hidden dim of the MLP classification head
    "dropout":       0.3,

    # Training
    "epochs":        400,
    "warmup_epochs": 2,      # epochs for linear LR warmup (then cosine annealing)
    "lr":            1e-3,
    "weight_decay":  1e-4,

    # Data augmentation (training only)
    "aug_hflip":     False,
    "aug_vflip":     False,
    "aug_rotation":  0,

    # W&B
    "wandb_project": "CMoE",
    "wandb_run":     "cmoe-octmnist-v1",

    # Resume training from a checkpoint saved in Drive.
    # Set to the full path of checkpoint.pt to continue a previous run,
    # or None to start fresh.
    "resume_ckpt":   f"{DRIVE_DIR}/checkpoint.pt",

    # If resuming, load optimizer state and LR from checkpoint, bypassing warmup.
    "resume_optimizer_state": True
}

# Part 2: Data Pipeline

This section converts raw MedMNIST images into graph-structured data. Each image is divided into fixed-size patches (nodes), and a k-NN graph is built from patch feature similarity. Data augmentation is applied during training only.

## Image-to-Graph Pipeline

`MedMNISTGraphPipeline` extracts non-overlapping patches from an image tensor and constructs a k-NN graph where each patch is a node.

In [ ]:
# The Image-to-Graph Pipeline
class MedMNISTGraphPipeline:
    def __init__(self, patch_size, k_neighbors):
        '''
        Args:
            patch_size (int): height or width of a patch
            k_neighbors (int): number of nearest neighbors to find around a node
        '''
        self.patch_size = patch_size
        self.k_neighbors = k_neighbors

    def extract_patches(self, img_tensor):
        c, h, w = img_tensor.shape
        p = self.patch_size

        assert h % p == 0 and w % p == 0, "Patch size not suitable."

        # Unfold into patches: [C, H/P, W/P, P, P]
        patches = img_tensor.unfold(1, p, p).unfold(2, p, p)
        num_patches = patches.shape[1] * patches.shape[2]

        # Reshape to [Num_Patches, C*P*P]
        patches = patches.contiguous().view(c, num_patches, p * p)
        patches = patches.permute(1, 0, 2).reshape(num_patches, -1)
        return patches

    def process(self, img_tensor):
        x = self.extract_patches(img_tensor)
        edge_index = knn_graph(x, k=self.k_neighbors, loop=False)
        return Data(x=x, edge_index=edge_index)

## Data Augmentation

`build_transform` assembles a torchvision augmentation pipeline from the config flags. It is applied only to the training split.

In [ ]:
# ── Build augmentation pipeline from config (training only) ───────────────────
def build_transform(config):
    aug_list = []
    # Add each augmentation only if enabled in config
    if config.get("aug_hflip", False):
        aug_list.append(transforms.RandomHorizontalFlip(p=0.5))
    if config.get("aug_vflip", False):
        aug_list.append(transforms.RandomVerticalFlip(p=0.5))
    if config.get("aug_rotation", 0) > 0:
        aug_list.append(transforms.RandomRotation(degrees=config["aug_rotation"]))
    # Return composed transform, or None if no augmentation is configured
    return transforms.Compose(aug_list) if aug_list else None


## Dataset Class

`NPZMedMNISTGraphDataset` wraps a MedMNIST `.npz` file. On each `__getitem__` call it converts the image to a PyG `Data` graph object, optionally applying augmentation for the training split.

In [ ]:
class NPZMedMNISTGraphDataset(Dataset):
    def __init__(self, npz_path, split, pipeline, transform=None):
        """
        Args:
            npz_path (str): Path to the MedMNIST .npz file.
            split (str): One of 'train', 'val', or 'test'.
            pipeline: The graph conversion pipeline.
            transform: Optional torchvision transforms applied before graph conversion.
                       Should only be passed for the training split.
        """
        self.pipeline = pipeline
        self.transform = transform

        data = np.load(npz_path)
        self.images = data[f'{split}_images']
        self.labels = data[f'{split}_labels']

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]
        label = self.labels[idx]

        img_tensor = torch.from_numpy(img).float()

        # Handle different image dimensions (grayscale vs. RGB)
        if img_tensor.ndim == 2:
            # Grayscale image (H, W) -> (C, H, W) with C=1
            img_tensor = img_tensor.unsqueeze(0)
        elif img_tensor.ndim == 3:
            # RGB image (H, W, C) -> (C, H, W)
            img_tensor = img_tensor.permute(2, 0, 1)
        else:
            raise ValueError(f"Unsupported image dimensions: {img_tensor.ndim}")

        # Normalize to [0, 1]
        img_tensor = img_tensor / 255.0

        # Apply augmentation (training only)
        if self.transform is not None:
            img_tensor = self.transform(img_tensor)

        graph_data = self.pipeline.process(img_tensor)

        label_val = label.item() if label.size == 1 else label[0]
        graph_data.y = torch.tensor([label_val], dtype=torch.long)

        return graph_data

## DataLoaders

Instantiate the pipeline, build all three dataset splits, and wrap them in PyG `DataLoader`s. Training data is shuffled; validation and test are not.

In [ ]:
# Initialize Pipeline
pipeline = MedMNISTGraphPipeline(patch_size=config["patch_size"], k_neighbors=config["k_neighbors"])

# Build augmentation transform from config (applied to training split only)
train_transform = build_transform(config)

# Initialize Datasets
train_dataset = NPZMedMNISTGraphDataset(npz_path=config["file_path"], split='train', pipeline=pipeline, transform=train_transform)
val_dataset   = NPZMedMNISTGraphDataset(npz_path=config["file_path"], split='val',   pipeline=pipeline)
test_dataset  = NPZMedMNISTGraphDataset(npz_path=config["file_path"], split='test',  pipeline=pipeline)

# Create PyG DataLoaders; shuffle train for better generalization
train_loader = DataLoader(train_dataset, batch_size=config["batch_size"], shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=config["batch_size"], shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=config["batch_size"], shuffle=False)


# Part 3: Model Architecture

This section defines all components of the CMoE (Classical Graph Mixture-of-Experts) model:
- **GlobalSummary** — pools node features into a single graph-level vector
- **ClassicalRouter** — uses a lightweight MLP to compute expert routing weights
- **Expert GNNs** — a diverse pool of graph neural networks (GCN, GAT, SAGE, GIN)
- **ClassicalMoELayer** — routes each sample through the top-k experts and aggregates outputs
- **NodeUpdater** — injects graph-level MoE output back into node features for iterative refinement
- **IterativeCMoE** — runs multiple GlobalSummary → ClassicalRouter → MoE passes
- **CMoEClassifier** — adds an attention pooling layer and MLP classification head

## 3.1 Global Summary

`GlobalSummary` collapses all node features in a graph into one fixed-size vector `g` using either mean pooling or learned attention pooling. This vector is the input to the classical router.

In [ ]:
class GlobalSummary(torch.nn.Module):
    """
    Produce a graph-level global vector g from node features x in a PyG Batch.
    """
    def __init__(self, in_dim: int, method: str="mean", attn_hidden: int=64):
        super().__init__()
        assert method in ["mean", "attn"]
        self.method = method

        if method == "attn":
            gate_nn = torch.nn.Sequential(
                torch.nn.Linear(in_dim, attn_hidden),
                torch.nn.ReLU(),
                torch.nn.Linear(attn_hidden, 1)
            )
            self.pool = GlobalAttention(gate_nn=gate_nn)

    def forward(self, batch_data):
        x = batch_data.x    # [total_nodes_in_batch, in_dim]
        batch = batch_data.batch  # [total_nodes_in_batch], graph id for each node
        if self.method == "mean":
            g = global_mean_pool(x, batch)  #[batch_size, in_dim]
        else:
            g = self.pool(x, batch)  #[batch_size, in_dim]
        return g

In [ ]:
# ── Derive feature_dim from data & instantiate GlobalSummary ──────────────────
# Fetch one mini-batch to determine node feature dimensionality (C * patch_size^2)
sample_batch = next(iter(train_loader))
feature_dim = sample_batch.x.shape[1]   # C * patch_size^2
print(f"feature_dim: {feature_dim}")

# Instantiate with pooling method from config
summary_module = GlobalSummary(
    in_dim      = feature_dim,
    method      = config["pool_method"],
    attn_hidden = config["attn_hidden"],
)


feature_dim: 16


## 3.2 Classical Router

`ClassicalRouter` maps the global summary vector `g` to expert routing weights via a two-layer MLP. A hidden layer with ReLU activation learns a non-linear projection from `feature_dim` to `n_experts`, and softmax normalizes the output into routing probabilities. This is a drop-in classical replacement for the quantum variational circuit router.

In [ ]:
class ClassicalRouter(torch.nn.Module):
    """
    Classical MLP routing layer for MoE.

    Flow:
      g [B, feature_dim]
        → Linear(feature_dim → router_hidden) → ReLU
        → Linear(router_hidden → n_experts)
        → Softmax
        → routing_weights [B, n_experts]
    """

    def __init__(self, feature_dim: int, n_experts: int, router_hidden: int = 64):
        super().__init__()
        self.router = torch.nn.Sequential(
            torch.nn.Linear(feature_dim, router_hidden),
            torch.nn.ReLU(),
            torch.nn.Linear(router_hidden, n_experts),
        )

    def forward(self, g: torch.Tensor) -> torch.Tensor:
        """
        Args:
            g: [B, feature_dim]  global graph vector from GlobalSummary
        Returns:
            routing_weights: [B, n_experts]  softmax routing probabilities
        """
        logits = self.router(g)                            # [B, n_experts]
        routing_weights = torch.softmax(logits, dim=-1)    # [B, n_experts], sums to 1
        return routing_weights


In [ ]:
# ── Smoke test ────────────────────────────────────────────────────────────────
classical_router = ClassicalRouter(
    feature_dim  = feature_dim,
    n_experts    = config["n_experts"],
    router_hidden = config["router_hidden"],
)

for batch in train_loader:
    g = summary_module(batch)                        # [B, feature_dim]
    routing_weights = classical_router(g)            # [B, n_experts]

    print("g shape:               ", g.shape)
    print("routing_weights shape: ", routing_weights.shape)
    print("routing_weights[0]:    ", routing_weights[0].detach().numpy())
    print("sum (should be 1.0):   ", routing_weights[0].sum().item())
    break


g shape:                torch.Size([32, 16])
routing_weights shape:  torch.Size([32, 8])
routing_weights[0]:     [0.13294482 0.12005121 0.11631238 0.1245738  0.10981984 0.13119683
 0.13024977 0.1348514 ]
sum (should be 1.0):    1.0


## 3.3 Expert GNNs

Each expert is a distinct 2-layer GNN architecture. Using four architectures (GCN, GAT, SAGE, GIN) in narrow and wide variants encourages specialization: different experts learn complementary structural patterns.

In [ ]:
class GCNExpert(torch.nn.Module):
    """2-layer Graph Convolutional Network."""
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, out_dim)

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)
        return global_mean_pool(x, batch)          # [B, out_dim]


In [ ]:
class GATExpert(torch.nn.Module):
    """2-layer Graph Attention Network."""
    def __init__(self, in_dim, hidden_dim, out_dim, heads=4):
        super().__init__()
        # First layer uses multi-head attention with concatenated outputs
        self.conv1 = GATConv(in_dim, hidden_dim, heads=heads, concat=True)
        # Second layer merges heads into a single output vector
        self.conv2 = GATConv(hidden_dim * heads, out_dim, heads=1, concat=False)

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)
        return global_mean_pool(x, batch)          # [B, out_dim]


In [ ]:
class SAGEExpert(torch.nn.Module):
    """2-layer GraphSAGE."""
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.conv1 = SAGEConv(in_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, out_dim)

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)
        return global_mean_pool(x, batch)          # [B, out_dim]


In [ ]:
class GINExpert(torch.nn.Module):
    """2-layer Graph Isomorphism Network."""
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        # Each GIN layer wraps an MLP as the aggregation function
        mlp1 = torch.nn.Sequential(torch.nn.Linear(in_dim, hidden_dim), torch.nn.ReLU(),
                                   torch.nn.Linear(hidden_dim, hidden_dim))
        mlp2 = torch.nn.Sequential(torch.nn.Linear(hidden_dim, out_dim), torch.nn.ReLU(),
                                   torch.nn.Linear(out_dim, out_dim))
        self.conv1 = GINConv(mlp1)
        self.conv2 = GINConv(mlp2)

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)
        return global_mean_pool(x, batch)          # [B, out_dim]


## 3.4 Classical MoE Layer

`ClassicalMoELayer` takes classical routing weights and runs only the top-k experts per sample. Their outputs are renormalized and averaged to produce the final MoE representation. Running all experts in parallel enables efficient batched GPU computation.

In [ ]:
# ── Classical MoE Layer ───────────────────────────────────────────────────────

class ClassicalMoELayer(torch.nn.Module):
    """
    Mixture-of-Experts layer driven by classical routing weights.

    Flow:
      batch_data  +  routing_weights [B, n_experts]
        → top-k selection & renormalization       → sparse weights [B, n_experts]
        → all experts process the graph           → [B, n_experts, out_dim]
        → weighted sum                            → [B, out_dim]
    """

    def __init__(self, experts: list, top_k: int = 2):
        super().__init__()
        assert top_k <= len(experts), "top_k cannot exceed the number of experts"
        self.experts  = torch.nn.ModuleList(experts)
        self.top_k    = top_k
        self.n_experts = len(experts)

    def forward(self, batch_data, routing_weights: torch.Tensor) -> torch.Tensor:
        """
        Args:
            batch_data:      PyG Batch (x, edge_index, batch)
            routing_weights: [B, n_experts] softmax probabilities from ClassicalRouter
        Returns:
            out: [B, out_dim]  top-k weighted combination of expert outputs
        """
        # ── 1. Top-k sparse gating ────────────────────────────────────────────
        topk_vals, topk_idx = torch.topk(routing_weights, self.top_k, dim=-1)  # [B, k]
        # Build a sparse weight matrix; zero out non-selected experts
        sparse_weights = torch.zeros_like(routing_weights)
        sparse_weights.scatter_(1, topk_idx, topk_vals)
        # Renormalize so selected weights still sum to 1
        sparse_weights = sparse_weights / sparse_weights.sum(dim=-1, keepdim=True)

        # ── 2. Run every expert on the graph batch ────────────────────────────
        # Stack → [B, n_experts, out_dim]
        expert_outs = torch.stack(
            [expert(batch_data.x, batch_data.edge_index, batch_data.batch)
             for expert in self.experts],
            dim=1
        )

        # ── 3. Weighted sum over experts ──────────────────────────────────────
        # sparse_weights: [B, n_experts] → unsqueeze → [B, n_experts, 1]
        out = (sparse_weights.unsqueeze(-1) * expert_outs).sum(dim=1)  # [B, out_dim]
        return out


In [ ]:
# ── Smoke test ────────────────────────────────────────────────────────────────

# Build expert pool — one per expert (n_experts = 8), varied configs for diversity
expert_pool = [
    GCNExpert (feature_dim, config["expert_hidden"],     config["expert_out"]),   # GCN  narrow
    GCNExpert (feature_dim, config["expert_hidden"] * 2, config["expert_out"]),   # GCN  wide
    GATExpert (feature_dim, config["expert_hidden"],     config["expert_out"], heads=4),  # GAT 4-head
    GATExpert (feature_dim, config["expert_hidden"],     config["expert_out"], heads=8),  # GAT 8-head
    SAGEExpert(feature_dim, config["expert_hidden"],     config["expert_out"]),   # SAGE narrow
    SAGEExpert(feature_dim, config["expert_hidden"] * 2, config["expert_out"]),   # SAGE wide
    GINExpert (feature_dim, config["expert_hidden"],     config["expert_out"]),   # GIN  narrow
    GINExpert (feature_dim, config["expert_hidden"] * 2, config["expert_out"]),   # GIN  wide
]
assert len(expert_pool) == config["n_experts"], "n_experts must equal len(expert_pool)"

moe_layer = ClassicalMoELayer(experts=expert_pool, top_k=config["top_k"])

for batch in train_loader:
    g               = summary_module(batch)               # [B, feature_dim]
    routing_weights = classical_router(g)                 # [B, 8]
    moe_out         = moe_layer(batch, routing_weights)   # [B, 128]

    print("routing_weights shape: ", routing_weights.shape)
    print("moe_out shape:         ", moe_out.shape)
    print("active experts/sample: ", config["top_k"], "of", config["n_experts"])
    break


routing_weights shape:  torch.Size([32, 8])
moe_out shape:          torch.Size([32, 128])
active experts/sample:  2 of 8


## 3.5 Node Updater

`NodeUpdater` feeds the graph-level MoE output back into the node feature space as a residual. Broadcasting the graph vector to every node and projecting it to `node_dim` lets each subsequent iteration operate on richer, MoE-informed features.

In [ ]:
class NodeUpdater(torch.nn.Module):
    """
    Updates node features using the graph-level MoE output.

    The MoE output [B, moe_dim] is broadcast to every node in its graph,
    projected to node_dim, then added as a residual to the current node features.
    LayerNorm stabilizes the mixed representation before the next iteration.

    Flow:
      x [total_nodes, node_dim]  +  moe_out [B, moe_dim]
        → broadcast:  moe_out[batch_idx]       → [total_nodes, moe_dim]
        → project:    Linear(moe_dim→node_dim) → [total_nodes, node_dim]
        → residual:   x + delta
        → LayerNorm                            → [total_nodes, node_dim]
    """

    def __init__(self, moe_dim: int, node_dim: int):
        super().__init__()
        self.proj = torch.nn.Linear(moe_dim, node_dim)
        self.norm = torch.nn.LayerNorm(node_dim)

    def forward(self, x: torch.Tensor, moe_out: torch.Tensor,
                batch_idx: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x:         [total_nodes, node_dim]  current node features
            moe_out:   [B, moe_dim]             graph-level MoE output
            batch_idx: [total_nodes]            graph id for each node (batch_data.batch)
        Returns:
            x_new: [total_nodes, node_dim]
        """
        moe_broadcast = moe_out[batch_idx]          # [total_nodes, moe_dim]
        delta  = self.proj(moe_broadcast)            # [total_nodes, node_dim]
        x_new  = self.norm(x + delta)               # [total_nodes, node_dim]
        return x_new

## 3.6 Iterative CMoE

`IterativeCMoE` chains the three core modules (GlobalSummary → ClassicalRouter → MoE) for `n_iterations` passes. After each pass except the last, `NodeUpdater` refines the node features so the next iteration sees progressively enriched representations.

In [ ]:
class IterativeCMoE(torch.nn.Module):
    """
    Iterative Classical Graph Mixture-of-Experts.

    Runs the full pipeline (GlobalSummary → ClassicalRouter → MoE) for
    n_iterations passes. After each pass (except the last), NodeUpdater
    injects the graph-level MoE output back into the node features so
    the next iteration operates on refined representations.

    Iteration flow (repeated n_iterations times):
      batch.x [total_nodes, node_dim]
        → GlobalSummary   → g [B, node_dim]
        → ClassicalRouter → routing_weights [B, n_experts]
        → MoE             → moe_out [B, expert_out]
        → NodeUpdater     → batch.x updated  (skipped on final iteration)

    Returns:
      moe_out: [B, expert_out]   output from the final iteration
      batch:   PyG Batch         with batch.x = refined node features from final iteration
    """

    def __init__(
        self,
        summary_module:   torch.nn.Module,
        classical_router: torch.nn.Module,
        moe_layer:        torch.nn.Module,
        node_updater:     torch.nn.Module,
        n_iterations:     int = 3,
    ):
        super().__init__()
        self.summary_module   = summary_module
        self.classical_router = classical_router
        self.moe_layer        = moe_layer
        self.node_updater     = node_updater
        self.n_iterations     = n_iterations

    def forward(self, batch_data):
        """
        Args:
            batch_data: PyG Batch (x, edge_index, batch, y)
        Returns:
            moe_out: [B, expert_out]  output from the final iteration
            batch:   PyG Batch        batch.x holds the final refined node features
        """
        # Clone to avoid mutating the original batch across DataLoader calls
        batch = batch_data.clone()

        for i in range(self.n_iterations):
            # Stage 1: summarise current node features into a global vector
            g = self.summary_module(batch)                       # [B, node_dim]

            # Stage 2: classical routing — maps global vector to expert weights
            routing_weights = self.classical_router(g)           # [B, n_experts]

            # Stage 3: mixture of experts processes the graph
            moe_out = self.moe_layer(batch, routing_weights)     # [B, expert_out]

            # Update node features for the next iteration (skip on last pass)
            if i < self.n_iterations - 1:
                batch.x = self.node_updater(
                    batch.x, moe_out, batch.batch                # [total_nodes, node_dim]
                )

        # Return both the MoE output and the batch carrying refined node features
        return moe_out, batch


In [ ]:
# ── Instantiation ─────────────────────────────────────────────────────────────
node_updater = NodeUpdater(
    moe_dim  = config["expert_out"],   # 128  — MoE output dim
    node_dim = feature_dim,            # must match node feature dim
)

iterative_cmoe = IterativeCMoE(
    summary_module   = summary_module,
    classical_router = classical_router,
    moe_layer        = moe_layer,
    node_updater     = node_updater,
    n_iterations     = config["n_iterations"],
)

# ── Smoke test ────────────────────────────────────────────────────────────────
for batch in train_loader:
    original_x_ptr = batch.x.data_ptr()

    moe_out, final_batch = iterative_cmoe(batch)   # unpack tuple

    print("n_iterations:             ", config["n_iterations"])
    print("moe_out shape:            ", moe_out.shape)        # [32, 128]
    print("final_batch.x shape:      ", final_batch.x.shape) # [total_nodes, feature_dim]
    assert batch.x.data_ptr() == original_x_ptr, "Original batch.x was mutated!"
    print("original batch.x safe:    True")
    break


n_iterations:              4
moe_out shape:             torch.Size([32, 128])
final_batch.x shape:       torch.Size([1568, 16])
original batch.x safe:    True


## 3.7 Full CMoE Classifier

`CMoEClassifier` wraps `IterativeCMoE` and adds a `GlobalAttention` pooling layer followed by a two-layer MLP head. The attention pooling learns to up-weight the most diagnostically relevant nodes before classification.

In [ ]:

class CMoEClassifier(torch.nn.Module):
    """
    Full CMoE pipeline ending in diagnostic classification.

    After the iterative loop, the refined node features are pooled with
    GlobalAttention, which learns to assign higher weight to the most
    diagnostically relevant nodes. The pooled vector is then classified
    by a two-layer MLP head.

    Flow:
      batch_data
        → IterativeCMoE (n_iterations passes)
            → final_batch.x  [total_nodes, node_dim]   refined node features
        → GlobalAttention pooling
            gate: Linear(node_dim → clf_hidden → 1)
            → attn_g  [B, node_dim]                    attention-weighted graph vector
        → MLP head
            Linear(node_dim → clf_hidden) → ReLU → Dropout → Linear(clf_hidden → n_classes)
            → logits  [B, n_classes]
    """

    def __init__(
        self,
        iterative_cmoe: torch.nn.Module,
        node_dim:   int,
        n_classes:  int,
        clf_hidden: int = 256,
        dropout:    float = 0.3,
    ):
        super().__init__()
        self.iterative_cmoe = iterative_cmoe

        # gate_nn scores each node → softmax over nodes in same graph → weighted sum
        gate_nn = torch.nn.Sequential(
            torch.nn.Linear(node_dim, clf_hidden),
            torch.nn.ReLU(),
            torch.nn.Linear(clf_hidden, 1),
        )
        self.attn_pool = GlobalAttention(gate_nn=gate_nn)

        self.classifier = torch.nn.Sequential(
            torch.nn.Linear(node_dim, clf_hidden),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(clf_hidden, n_classes),
        )

    def forward(self, batch_data) -> torch.Tensor:
        _, final_batch = self.iterative_cmoe(batch_data)
        attn_g = self.attn_pool(final_batch.x, final_batch.batch)  # [B, node_dim]
        logits = self.classifier(attn_g)                            # [B, n_classes]
        return logits


In [ ]:
# ── Instantiation + Smoke test ────────────────────────────────────────────────
model = CMoEClassifier(
    iterative_cmoe = iterative_cmoe,
    node_dim       = feature_dim,
    n_classes      = config["n_classes"],
    clf_hidden     = config["clf_hidden"],
    dropout        = 0.3,
)

for batch in train_loader:
    logits = model(batch)                          # [B, n_classes]
    preds  = logits.argmax(dim=-1)                 # [B]

    print("logits shape:  ", logits.shape)
    print("preds sample:  ", preds[:8].tolist())
    print("labels sample: ", batch.y[:8].squeeze().tolist())

    logits.sum().backward()
    assert model.attn_pool.gate_nn[0].weight.grad is not None, "attn_pool has no gradient"
    print("gradients flow: True")
    break


logits shape:   torch.Size([32, 4])
preds sample:   [0, 0, 0, 0, 0, 0, 1, 0]
labels sample:  [0, 0, 2, 3, 0, 0, 2, 2]
gradients flow: True


/tmp/ipykernel_6875/3964591183.py:39: UserWarning: 'nn.glob.GlobalAttention' is deprecated, use 'nn.aggr.AttentionalAggregation' instead
  self.attn_pool = GlobalAttention(gate_nn=gate_nn)


In [ ]:
# Print a per-module parameter breakdown to understand model size
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"{'Module':<60} {'Parameters':>12}")
print("-" * 74)
for name, module in model.named_modules():
    params = sum(p.numel() for p in module.parameters(recurse=False))
    if params > 0:
        print(f"{name:<60} {params:>12,}")
print("-" * 74)
print(f"{'Total parameters':<60} {total_params:>12,}")
print(f"{'Trainable parameters':<60} {trainable_params:>12,}")
print(f"{'Non-trainable parameters':<60} {total_params - trainable_params:>12,}")

Module                                                         Parameters
--------------------------------------------------------------------------
iterative_cmoe.classical_router.router.0                            1,088
iterative_cmoe.classical_router.router.2                              520
iterative_cmoe.moe_layer.experts.0.conv1                               64
iterative_cmoe.moe_layer.experts.0.conv1.lin                        1,024
iterative_cmoe.moe_layer.experts.0.conv2                              128
iterative_cmoe.moe_layer.experts.0.conv2.lin                        8,192
iterative_cmoe.moe_layer.experts.1.conv1                              128
iterative_cmoe.moe_layer.experts.1.conv1.lin                        2,048
iterative_cmoe.moe_layer.experts.1.conv2                              128
iterative_cmoe.moe_layer.experts.1.conv2.lin                       16,384
iterative_cmoe.moe_layer.experts.2.conv1                              768
iterative_cmoe.moe_layer.experts.2.co

# Part 4: Training

This section sets up Weights & Biases logging, defines the per-epoch training and evaluation helpers, and runs the full training loop. The loop uses linear LR warmup followed by cosine annealing, saves checkpoints to Drive every epoch, and supports resuming from any saved checkpoint.

In [ ]:
import wandb

# Log in to Weights & Biases for experiment tracking
wandb.login(key="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")


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: peijingl1221 (peijingl1221-carnegie-mellon-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    """Run one full pass over the training set and return loss and accuracy."""
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for batch in loader:
        batch  = batch.to(device)
        optimizer.zero_grad()
        logits = model(batch)
        labels = batch.y.squeeze()
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        # Accumulate weighted loss and correct predictions for the epoch average
        total_loss += loss.item() * batch.num_graphs
        correct    += (logits.argmax(dim=-1) == labels).sum().item()
        total      += batch.num_graphs
    return total_loss / total, correct / total


In [ ]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """Evaluate the model on a given loader (validation or test) without gradients."""
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_probs, all_labs = [], []
    for batch in loader:
        batch  = batch.to(device)
        logits = model(batch)
        labels = batch.y.squeeze()
        loss   = criterion(logits, labels)
        total_loss += loss.item() * batch.num_graphs
        correct    += (logits.argmax(dim=-1) == labels).sum().item()
        total      += batch.num_graphs
        all_probs.append(torch.softmax(logits, dim=-1).cpu())
        all_labs.append(labels.cpu())
    all_probs = torch.cat(all_probs, dim=0).numpy()
    all_labs  = torch.cat(all_labs,  dim=0).numpy()
    try:
        auc = roc_auc_score(all_labs, all_probs, multi_class="ovr", average="macro")
    except ValueError:
        auc = float("nan")
    return total_loss / total, correct / total, auc

In [ ]:
def train(model, train_loader, val_loader, config):
    """
    Train CMoEClassifier with W&B logging, full resume support,
    and linear LR warmup followed by cosine annealing decay.
    Checkpoints and history are saved to Google Drive.
    """
    device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model     = model.to(device)
    criterion = torch.nn.CrossEntropyLoss()

    # Initialize optimizer with the target_lr. This will be the base_lr that LinearLR scales.
    target_lr = config["lr"]
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr           = target_lr,
        weight_decay = config["weight_decay"],
    )

    # Checkpoints and history go to Drive so they survive kernel restarts
    best_ckpt    = f"{DRIVE_DIR}/best_model.pt"
    last_ckpt    = f"{DRIVE_DIR}/checkpoint.pt"
    history_path = f"{DRIVE_DIR}/history.json"

    # ── Initial state variables ────────────────────────────────────────────────
    start_epoch  = 1
    best_val_auc = 0.0
    wandb_run_id = None
    history      = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "train_auc": [], "val_auc": [], "lr": []}
    warmup_epochs_to_run = config.get("warmup_epochs", 5)

    # Determine if resuming from the best model is requested and possible
    should_resume_from_best = config.get("resume_ckpt") is not None and os.path.exists(best_ckpt)

    if should_resume_from_best:
        print(f"Resuming training from '{best_ckpt}' (best validation AUC).")
        model.load_state_dict(torch.load(best_ckpt, map_location=device))

        # Load history to find when best model was saved and its AUC
        if os.path.exists(history_path):
            with open(history_path) as f:
                loaded_history = json.load(f)
            history = loaded_history

            if loaded_history["val_auc"]:
                best_epoch_idx = np.argmax(loaded_history["val_auc"])
                start_epoch = best_epoch_idx + 1
                best_val_auc = loaded_history["val_auc"][best_epoch_idx]
                print(f"  (Best model achieved AUC {best_val_auc:.4f} at epoch {best_epoch_idx + 1}).")
            else:
                print("  Warning: No validation AUC history found. Starting from epoch 1.")
                start_epoch = 1
        else:
            print(f"  Warning: History file '{history_path}' not found. Starting from epoch 1.")
            start_epoch = 1

        print(f"Starting new training phase from epoch {start_epoch} with LR warmup from 0.00e+00 to {target_lr:.2e}.")
        wandb_run_id = None

    else:
        print(f"Starting fresh training run with LR warmup from 0.00e+00 to {target_lr:.2e}.")
        start_epoch = 1

    # ── Rebuild scheduler (always linear warmup from 0 to target_lr, then cosine annealing) ────
    epochs_for_scheduler = config["epochs"] - (start_epoch - 1)
    effective_warmup_epochs = min(warmup_epochs_to_run, epochs_for_scheduler)
    effective_cosine_epochs = max(0, epochs_for_scheduler - effective_warmup_epochs)

    warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
        optimizer, start_factor=1e-6, end_factor=1.0, total_iters=effective_warmup_epochs,
    )
    cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=effective_cosine_epochs
    )
    scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[effective_warmup_epochs],
    )
    print(f"Scheduler: {effective_warmup_epochs} warmup epochs, then {effective_cosine_epochs} cosine annealing epochs.")

    # ── W&B init ─────────────────────────────────
    wandb.init(
        project=config["wandb_project"], name=config["wandb_run"],
        id=wandb_run_id, resume="allow", config=config,
    )
    wandb.watch(model, log="gradients", log_freq=10)

    for epoch in range(start_epoch, config["epochs"] + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss,   val_acc,  val_auc  = evaluate(model, val_loader, criterion, device)
        scheduler.step()
        current_lr = scheduler.get_last_lr()[0]

        # Append to history
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["val_auc"].append(val_auc)
        history["lr"].append(current_lr)

        wandb.log(
            {
            "epoch": epoch, "train/loss": train_loss, "train/acc": train_acc,
            "val/loss": val_loss, "val/acc": val_acc, "val/auc": val_auc, "lr": current_lr,
        })

        phase = "warmup" if (epoch - start_epoch + 1) <= warmup_epochs_to_run else "cosine"
        print(
            f"Epoch {epoch:03d}/{config['epochs']} [{phase}] | "
            f"lr {current_lr:.2e} | "
            f"train loss {train_loss:.4f}  acc {train_acc:.4f} | "
            f"val loss {val_loss:.4f}  acc {val_acc:.4f}  auc {val_auc:.4f}"
        )

        # Save checkpoint and history to Drive every epoch
        torch.save(
            {
            "epoch": epoch, "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "best_val_auc": best_val_auc, "wandb_run_id": wandb.run.id,
        }, last_ckpt)
        with open(history_path, "w") as f:
            json.dump(history, f)

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            torch.save(model.state_dict(), best_ckpt)
            wandb.save(best_ckpt)
            print(f"  new best val auc: {best_val_auc:.4f}  ->  saved to Drive")

    best_val_acc = max(history["val_acc"]) if history["val_acc"] else float("nan")
    wandb.log({"best_val_acc": best_val_acc, "best_val_auc": best_val_auc})
    wandb.finish()
    print(f"\nTraining complete. Best val auc: {best_val_auc:.4f}")
    print(f"Checkpoints saved in: {DRIVE_DIR}")
    return model, history

In [ ]:
# ── Run training ──────────────────────────────────────────────────────────────
model, history = train(model, train_loader, val_loader, config)


Starting fresh training run with LR warmup from 0.00e+00 to 1.00e-03.
Scheduler: 2 warmup epochs, then 398 cosine annealing epochs.


wandb: WARNING Saving files without folders. If you want to preserve subdirectories pass base_path to wandb.save, i.e. wandb.save("/mnt/folder/file.h5", base_path="/mnt")
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


Epoch 001/400 [warmup] | lr 5.00e-04 | train loss 1.3515  acc 0.3453 | val loss 1.3413  acc 0.3424  auc 0.5460
  new best val auc: 0.5460  ->  saved to Drive
Epoch 002/400 [warmup] | lr 1.00e-03 | train loss 1.0100  acc 0.5995 | val loss 0.9279  acc 0.6553  auc 0.7632
  new best val auc: 0.7632  ->  saved to Drive
Epoch 003/400 [cosine] | lr 1.00e-03 | train loss 0.8412  acc 0.6930 | val loss 0.7935  acc 0.7122  auc 0.8187
  new best val auc: 0.8187  ->  saved to Drive
Epoch 004/400 [cosine] | lr 1.00e-03 | train loss 0.7765  acc 0.7190 | val loss 0.7451  acc 0.7285  auc 0.8387
  new best val auc: 0.8387  ->  saved to Drive
Epoch 005/400 [cosine] | lr 1.00e-03 | train loss 0.7493  acc 0.7296 | val loss 0.7449  acc 0.7329  auc 0.8386
Epoch 006/400 [cosine] | lr 1.00e-03 | train loss 0.7327  acc 0.7359 | val loss 0.7203  acc 0.7441  auc 0.8478
  new best val auc: 0.8478  ->  saved to Drive
Epoch 007/400 [cosine] | lr 1.00e-03 | train loss 0.7172  acc 0.7429 | val loss 0.7070  acc 0.7489 

best_val_acc,▁
best_val_auc,▁
epoch,▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇███
lr,███████▇▇▇▇▆▆▆▆▅▅▅▅▅▅▅▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁
train/acc,▁▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇████████████████████████
train/loss,██▇▇▇▆▅▅▅▅▅▅▅▅▄▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁
val/acc,▁▅▅▅▅▆▆▆▆▆▅▆▇▇▇▇▆▇▇▇▇▇▇█▇██▇▇██▇████████
val/auc,▁▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇█▇███████████████████
val/loss,██▇▇▇▆▅▅▅▅▄▃▃▃▄▃▃▂▅▃▂▂▂▂▃▂▁▂▁▁▂▁▁▁▁▁▁▁▁▁
best_val_acc,0.8149
best_val_auc,0.90945



Training complete. Best val auc: 0.9095
Checkpoints saved in: /content/drive/MyDrive/CMoE


In [ ]:
# ── Test Evaluation (best checkpoint) ────────────────────────────────────────
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
best_ckpt = f"{DRIVE_DIR}/best_model.pt"

model.load_state_dict(torch.load(best_ckpt, map_location=device))
model.eval().to(device)

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for batch in test_loader:
        batch  = batch.to(device)
        logits = model(batch)
        probs  = torch.softmax(logits, dim=-1).cpu()
        preds  = logits.argmax(dim=-1).cpu().tolist()
        labs   = batch.y.squeeze().cpu().tolist()
        all_preds.extend(preds)
        all_labels.extend(labs if isinstance(labs, list) else [labs])
        all_probs.append(probs)

all_probs = torch.cat(all_probs, dim=0).numpy()
test_acc  = (np.array(all_preds) == np.array(all_labels)).mean() * 100
try:
    test_auc = roc_auc_score(all_labels, all_probs, multi_class="ovr", average="macro")
except ValueError:
    test_auc = float("nan")

print(f"Test Accuracy : {test_acc:.2f}%")
print(f"Test Macro AUC: {test_auc:.4f}")
print()
print(classification_report(
    all_labels, all_preds,
    target_names=[f"Class {i}" for i in range(config["n_classes"])]
))


# Part 5: Evaluation & Visualization

This section loads the saved training history, plots loss/accuracy/LR curves, loads the best model checkpoint and runs it on the held-out test set, then displays a confusion matrix and a per-class classification report.

In [ ]:
# ── Load history (works even if kernel was restarted after training) ──────────
history_path = f"{DRIVE_DIR}/history.json"
if 'history' not in dir() or not history["train_loss"]:
    with open(history_path) as f:
        history = json.load(f)

epochs = range(1, len(history["train_loss"]) + 1)
warmup_end = config["warmup_epochs"]

# ── Figure layout ─────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.38, wspace=0.32)

ax_loss = fig.add_subplot(gs[0, 0])
ax_acc  = fig.add_subplot(gs[0, 1])
ax_lr   = fig.add_subplot(gs[0, 2])
ax_cm   = fig.add_subplot(gs[1, :2])
ax_rep  = fig.add_subplot(gs[1, 2])
ax_rep.axis("off")

# ── Helper: shade warmup region ───────────────────────────────────────────────
def shade_warmup(ax):
    ax.axvspan(1, warmup_end + 0.5, alpha=0.08, color="orange", label="warmup")
    ax.axvline(warmup_end + 0.5, color="orange", lw=0.8, ls="--")

# ── 1. Loss curves ────────────────────────────────────────────────────────────
ax_loss.plot(epochs, history["train_loss"], label="train", lw=2)
ax_loss.plot(epochs, history["val_loss"],   label="val",   lw=2, ls="--")
shade_warmup(ax_loss)
ax_loss.set_title("Loss")
ax_loss.set_xlabel("Epoch")
ax_loss.set_ylabel("Cross-entropy loss")
ax_loss.legend()
ax_loss.grid(alpha=0.3)

# ── 2. Accuracy curves ────────────────────────────────────────────────────────
ax_acc.plot(epochs, [a * 100 for a in history["train_acc"]], label="train", lw=2)
ax_acc.plot(epochs, [a * 100 for a in history["val_acc"]],   label="val",   lw=2, ls="--")
best_epoch = int(np.argmax(history["val_acc"])) + 1
best_acc   = max(history["val_acc"]) * 100
ax_acc.axvline(best_epoch, color="green", lw=1, ls=":", label=f"best val ({best_acc:.1f}%)")
shade_warmup(ax_acc)
ax_acc.set_title("Accuracy")
ax_acc.set_xlabel("Epoch")
ax_acc.set_ylabel("Accuracy (%)")
ax_acc.legend()
ax_acc.grid(alpha=0.3)

# ── 3. AUC curve (top-right panel) ───────────────────────────────────────────
if history.get("val_auc"):
    ax_lr.plot(epochs[:len(history["val_auc"])], history["val_auc"], color="teal", lw=2)
    shade_warmup(ax_lr)
    best_auc_ep = int(np.argmax(history["val_auc"])) + 1
    ax_lr.axvline(best_auc_ep, color="teal", lw=1, ls=":", label=f"best ({max(history['val_auc']):.4f})")
    ax_lr.set_title("Val Macro AUC")
    ax_lr.set_xlabel("Epoch")
    ax_lr.set_ylabel("AUC")
    ax_lr.legend()
else:
    ax_lr.plot(epochs, history["lr"], color="purple", lw=2)
    shade_warmup(ax_lr)
    ax_lr.set_title("Learning Rate Schedule")
    ax_lr.set_xlabel("Epoch")
    ax_lr.set_ylabel("LR")
    ax_lr.set_yscale("log")
ax_lr.grid(alpha=0.3)

# ── 4. Confusion matrix on test set (best model) ──────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
best_ckpt = f"{DRIVE_DIR}/best_model.pt"
model.load_state_dict(torch.load(best_ckpt, map_location=device))
model.eval().to(device)

all_preds, all_labels = [], []
with torch.no_grad():
    for batch in val_loader:
        batch = batch.to(device)
        preds = model(batch).argmax(dim=-1).cpu().tolist()
        labs  = batch.y.squeeze().cpu().tolist()
        all_preds.extend(preds)
        all_labels.extend(labs if isinstance(labs, list) else [labs])

cm = confusion_matrix(all_labels, all_preds)
val_acc = (np.array(all_preds) == np.array(all_labels)).mean() * 100
all_probs_np = torch.cat([
    torch.softmax(model(batch.to(device)), dim=-1).cpu()
    for batch in val_loader
], dim=0).detach().numpy()
try:
    val_auc = roc_auc_score(all_labels, all_probs_np, multi_class="ovr", average="macro")
except ValueError:
    val_auc = float("nan")
print(f"Val Accuracy: {val_acc:.2f}%  |  Val Macro AUC: {val_auc:.4f}")

im = ax_cm.imshow(cm, interpolation="nearest", cmap="Blues")
fig.colorbar(im, ax=ax_cm, fraction=0.046, pad=0.04)
tick_marks = np.arange(config["n_classes"])
ax_cm.set_xticks(tick_marks)
ax_cm.set_yticks(tick_marks)
ax_cm.set_xticklabels([f"Pred {i}" for i in tick_marks])
ax_cm.set_yticklabels([f"True {i}" for i in tick_marks])
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax_cm.text(j, i, str(cm[i, j]), ha="center", va="center",
                   color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=13)
ax_cm.set_title(f"Confusion Matrix — Val Acc: {val_acc:.1f}%  AUC: {val_auc:.4f}")
ax_cm.set_ylabel("True label")
ax_cm.set_xlabel("Predicted label")

# ── 5. Classification report ──────────────────────────────────────────────────
report = classification_report(all_labels, all_preds,
                               target_names=[f"Class {i}" for i in range(config["n_classes"])])
ax_rep.text(0.02, 0.98, report, transform=ax_rep.transAxes,
            fontsize=9, verticalalignment="top", fontfamily="monospace",
            bbox=dict(boxstyle="round", facecolor="whitesmoke", alpha=0.8))
ax_rep.set_title("Classification Report")

plt.suptitle(f"CMoE Results — {config['wandb_run']}  |  Val Acc {val_acc:.1f}%  AUC {val_auc:.4f}", fontsize=13, y=1.01)

plot_path = f"{DRIVE_DIR}/results.png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Plot saved to: {plot_path}")